# Copyright (c) 2026 Khush Patil. All Rights Reserved.
# This code is for personal/educational viewing only. 
# Unauthorized copying, modification, or distribution is strictly prohibited.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import timm
import os

# --- 1. Setup Device & Transforms ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 2. Auto-Detect and Load Datasets ---
train_dir = None
test_dir = None

# Scan Colab storage to find exactly where the folders unzipped
for root, dirs, files in os.walk('/content'):
    for d in dirs:
        if d.lower() == 'train':
            train_dir = os.path.join(root, d)
        elif d.lower() in ['test', 'val', 'validation']:
            test_dir = os.path.join(root, d)

if not train_dir or not test_dir:
    raise FileNotFoundError("❌ Could not find Train or Test folders. Double check the unzip command!")

print(f"✅ Auto-Detected Train Path: {train_dir}")
print(f"✅ Auto-Detected Test Path: {test_dir}")
print("-" * 50)

# Load the data using the exact same transform
train_data = datasets.ImageFolder(root=train_dir, transform=transform)
test_data = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

num_classes = len(train_data.classes)

# CRITICAL: Print this list and copy it for your app.py later!
print("COPY THIS LIST FOR YOUR FASTAPI MAIN.PY:")
print(train_data.classes)
print("-" * 50)

# --- 3. Define the Models ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=num_classes)

class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        resnet = timm.create_model('resnet18', pretrained=True)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn_backbone(x)
        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

# --- 4. Training & Evaluation Functions ---
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def train_model(model, name, epochs=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"--- Training {name} ---")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Evaluate on test set after each epoch
        test_acc = evaluate_model(model, test_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Test Accuracy: {test_acc:.2f}%")

    # Save the model
    os.makedirs('models', exist_ok=True)
    save_path = f'models/{name}.pth'
    torch.save(model.state_dict(), save_path)
    print(f"Saved {name} to {save_path}\n")

# --- 5. Execute ---
cnn_model = SimpleCNN()
hybrid_model = HybridModel()

# Naming them _sans so they don't overwrite your Devanagari models
train_model(cnn_model, "cnn_sans", epochs=5)
train_model(vit_model, "vit_sans", epochs=3)
train_model(hybrid_model, "hybrid_sans", epochs=4)

✅ Auto-Detected Train Path: /content/Sanskrit/train
✅ Auto-Detected Test Path: /content/Sanskrit/validation
--------------------------------------------------
COPY THIS LIST FOR YOUR APP.PY:
['bra', 'dda', 'dma', 'dra', 'dwa', 'dya', 'gri', 'hra', 'hri', 'hya', 'kra', 'mra', 'mri', 'pra', 'rhi', 'shra', 'tra', 'traw', 'tta']
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

--- Training cnn_sans ---
Epoch 1/5 | Loss: 3.3473 | Test Accuracy: 19.14%
Epoch 2/5 | Loss: 2.4248 | Test Accuracy: 40.43%
Epoch 3/5 | Loss: 2.1084 | Test Accuracy: 52.15%
Epoch 4/5 | Loss: 1.8653 | Test Accuracy: 52.63%
Epoch 5/5 | Loss: 1.6512 | Test Accuracy: 65.31%
Saved cnn_sans to models/cnn_sans.pth

--- Training vit_sans ---
Epoch 1/3 | Loss: 3.1283 | Test Accuracy: 6.94%
Epoch 2/3 | Loss: 3.0096 | Test Accuracy: 6.22%
Epoch 3/3 | Loss: 2.9880 | Test Accuracy: 5.26%
Saved vit_sans to models/vit_sans.pth

--- Training hybrid_sans ---
Epoch 1/8 | Loss: 1.6064 | Test Accuracy: 87.56%
Epoch 2/8 | Loss: 0.2421 | Test Accuracy: 96.17%
Epoch 3/8 | Loss: 0.1726 | Test Accuracy: 97.61%
Epoch 4/8 | Loss: 0.1210 | Test Accuracy: 95.93%
Epoch 5/8 | Loss: 0.0966 | Test Accuracy: 95.22%
Epoch 6/8 | Loss: 0.1055 | Test Accuracy: 94.98%
Epoch 7/8 | Loss: 0.1225 | Test Accuracy: 97.37%
Epoch 8/8 | Loss: 0.1110 | Test Accuracy: 95.22%
Saved hybrid_sans to models/hybrid_sans.pth

